# 02. 데이터 분석 Tool 함수 정의

## 실험 목표
- 에이전트에 등록할 pandas 기반 **Tool 함수** 5종 설계·구현
- 각 Tool 함수 단독 실행 테스트 (에이전트 없이)
- `@agent.tool` 데코레이터로 에이전트에 등록하는 방법 이해

## 구현할 Tool 목록
| Tool 함수 | 기능 |
|---|---|
| `describe_data` | 기본 통계량 요약 |
| `check_missing` | 결측치 현황 분석 |
| `correlation_matrix` | 수치형 컬럼 상관관계 |
| `detect_outliers` | IQR 기반 이상치 탐지 |
| `plot_distribution` | 컬럼 분포 시각화 |

---
## 0. 환경 준비

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.gemini import GeminiModel
from dataclasses import dataclass

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# 샘플 데이터 로드
df = pd.read_csv("data/sample_data.csv")
print(f"✅ 데이터 로드 완료: {df.shape} (행 x 열)")
df.head()

---
## 1. Tool 함수 구현

In [ ]:
# ────────────────────────────────────────────
# Tool 1: describe_data — 기본 통계량 요약
# ────────────────────────────────────────────
def describe_data(df: pd.DataFrame) -> dict:
    """
    데이터프레임의 기본 통계량을 반환합니다.
    수치형 컬럼: mean, std, min, max, 25/50/75 percentile
    범주형 컬럼: 유니크 값 수, 최빈값
    """
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    
    result = {
        "shape": {"rows": int(df.shape[0]), "columns": int(df.shape[1])},
        "numeric_summary": df[numeric_cols].describe().round(2).to_dict() if numeric_cols else {},
        "categorical_summary": {
            col: {
                "unique_count": int(df[col].nunique()),
                "top_value": str(df[col].mode()[0]) if not df[col].mode().empty else None,
                "top_freq": int(df[col].value_counts().iloc[0]) if not df[col].value_counts().empty else 0
            }
            for col in cat_cols
        }
    }
    return result

# 단독 테스트
desc = describe_data(df)
print("[describe_data 테스트]")
print(f"데이터 형태: {desc['shape']}")
print(f"수치형 컬럼: {list(desc['numeric_summary'].keys())}")
print(f"범주형 컬럼: {list(desc['categorical_summary'].keys())}")

In [ ]:
# ────────────────────────────────────────────
# Tool 2: check_missing — 결측치 현황 분석
# ────────────────────────────────────────────
def check_missing(df: pd.DataFrame) -> dict:
    """
    각 컬럼의 결측치 개수와 비율을 반환합니다.
    결측치가 없는 컬럼은 결과에서 제외합니다.
    """
    missing_count = df.isnull().sum()
    missing_ratio = (df.isnull().sum() / len(df) * 100).round(2)
    
    missing_df = pd.DataFrame({
        'count': missing_count,
        'ratio(%)': missing_ratio
    }).query('count > 0').sort_values('count', ascending=False)
    
    result = {
        "total_missing_cells": int(df.isnull().sum().sum()),
        "columns_with_missing": missing_df.to_dict(orient='index')
    }
    return result

# 단독 테스트
missing = check_missing(df)
print("[check_missing 테스트]")
print(f"총 결측 셀 수: {missing['total_missing_cells']}")
print("결측치 있는 컬럼:")
for col, info in missing['columns_with_missing'].items():
    print(f"  {col}: {info['count']}개 ({info['ratio(%)']}%)")

In [ ]:
# ────────────────────────────────────────────
# Tool 3: correlation_matrix — 상관관계 분석
# ────────────────────────────────────────────
def correlation_matrix(df: pd.DataFrame, method: str = 'pearson') -> dict:
    """
    수치형 컬럼 간 상관관계 행렬을 계산하고 히트맵을 출력합니다.
    method: 'pearson' | 'spearman' | 'kendall'
    """
    numeric_df = df.select_dtypes(include='number')
    corr = numeric_df.corr(method=method).round(3)
    
    # 절댓값 기준 상위 상관 쌍 추출
    corr_pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            col_a = corr.columns[i]
            col_b = corr.columns[j]
            val = corr.iloc[i, j]
            corr_pairs.append((col_a, col_b, val))
    corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    
    # 히트맵 시각화
    fig, ax = plt.subplots(figsize=(8, 6))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0, ax=ax,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title(f'상관관계 히트맵 ({method})', fontsize=13, pad=15)
    plt.tight_layout()
    plt.show()
    
    return {
        "method": method,
        "top_correlations": [
            {"col_a": a, "col_b": b, "corr": float(v)}
            for a, b, v in corr_pairs[:5]
        ],
        "matrix": corr.to_dict()
    }

# 단독 테스트
corr_result = correlation_matrix(df)
print("\n[상위 5개 상관쌍]")
for pair in corr_result['top_correlations']:
    print(f"  {pair['col_a']} <-> {pair['col_b']}: {pair['corr']}")

In [ ]:
# ────────────────────────────────────────────
# Tool 4: detect_outliers — IQR 기반 이상치 탐지
# ────────────────────────────────────────────
def detect_outliers(df: pd.DataFrame, column: str) -> dict:
    """
    IQR(사분위수 범위)를 이용하여 특정 컬럼의 이상치를 탐지합니다.
    기준: Q1 - 1.5*IQR 미만 또는 Q3 + 1.5*IQR 초과
    """
    if column not in df.columns:
        return {"error": f"'{column}' 컬럼이 존재하지 않습니다."}
    if not pd.api.types.is_numeric_dtype(df[column]):
        return {"error": f"'{column}'은 수치형 컬럼이 아닙니다."}
    
    series = df[column].dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outlier_mask = (df[column] < lower) | (df[column] > upper)
    outlier_indices = df[outlier_mask].index.tolist()
    outlier_values = df.loc[outlier_mask, column].tolist()
    
    # 박스플롯 시각화
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    series.plot(kind='box', ax=axes[0], color='steelblue')
    axes[0].set_title(f'{column} 박스플롯')
    axes[0].axhline(lower, color='red', linestyle='--', linewidth=1, label=f'하한: {lower:.2f}')
    axes[0].axhline(upper, color='red', linestyle='--', linewidth=1, label=f'상한: {upper:.2f}')
    axes[0].legend(fontsize=8)
    
    series.plot(kind='hist', bins=30, ax=axes[1], color='steelblue', alpha=0.7)
    axes[1].axvline(lower, color='red', linestyle='--', linewidth=1)
    axes[1].axvline(upper, color='red', linestyle='--', linewidth=1)
    axes[1].set_title(f'{column} 히스토그램 (이상치 경계)')
    
    plt.tight_layout()
    plt.show()
    
    return {
        "column": column,
        "Q1": round(float(Q1), 3),
        "Q3": round(float(Q3), 3),
        "IQR": round(float(IQR), 3),
        "lower_bound": round(float(lower), 3),
        "upper_bound": round(float(upper), 3),
        "outlier_count": len(outlier_indices),
        "outlier_ratio(%)": round(len(outlier_indices) / len(series) * 100, 2),
        "outlier_values_sample": [round(float(v), 3) for v in outlier_values[:10]]
    }

# 단독 테스트
outlier_result = detect_outliers(df, 'Fare')
print("[detect_outliers 테스트 - Fare 컬럼]")
for k, v in outlier_result.items():
    print(f"  {k}: {v}")

In [ ]:
# ────────────────────────────────────────────
# Tool 5: plot_distribution — 컬럼 분포 시각화
# ────────────────────────────────────────────
def plot_distribution(df: pd.DataFrame, column: str) -> dict:
    """
    컬럼의 분포를 시각화합니다.
    - 수치형: 히스토그램 + KDE 곡선
    - 범주형: 막대그래프
    """
    if column not in df.columns:
        return {"error": f"'{column}' 컬럼이 존재하지 않습니다."}
    
    series = df[column].dropna()
    is_numeric = pd.api.types.is_numeric_dtype(series)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    if is_numeric:
        series.plot(kind='hist', bins=30, density=True,
                    color='steelblue', alpha=0.6, ax=ax, label='히스토그램')
        series.plot(kind='kde', color='darkblue', linewidth=2, ax=ax, label='KDE')
        ax.axvline(series.mean(), color='red', linestyle='--',
                   linewidth=1.5, label=f'평균: {series.mean():.2f}')
        ax.axvline(series.median(), color='orange', linestyle='-.',
                   linewidth=1.5, label=f'중앙값: {series.median():.2f}')
        ax.legend()
        summary = {
            "type": "numeric",
            "mean": round(float(series.mean()), 3),
            "median": round(float(series.median()), 3),
            "std": round(float(series.std()), 3),
            "skewness": round(float(series.skew()), 3)
        }
    else:
        value_counts = series.value_counts().head(10)
        value_counts.plot(kind='bar', color='steelblue', ax=ax, rot=30)
        ax.set_ylabel('빈도')
        summary = {
            "type": "categorical",
            "unique_count": int(series.nunique()),
            "top_values": value_counts.head(5).to_dict()
        }
    
    ax.set_title(f'{column} 분포', fontsize=13)
    ax.set_xlabel(column)
    plt.tight_layout()
    plt.show()
    
    return summary

# 수치형 테스트
print("[plot_distribution 테스트 - Age (수치형)]")
result_age = plot_distribution(df, 'Age')
print(result_age)

In [ ]:
# 범주형 테스트
print("[plot_distribution 테스트 - Sex (범주형)]")
result_sex = plot_distribution(df, 'Sex')
print(result_sex)

---
## 2. 에이전트에 Tool 등록

In [ ]:
# 에이전트에 전달할 데이터프레임을 Dependency로 설정
@dataclass
class DataDeps:
    df: pd.DataFrame
    csv_path: str

model = GeminiModel(model_name="gemini-2.0-flash", api_key=api_key)

da_agent = Agent(
    model=model,
    deps_type=DataDeps,
    system_prompt="""
    당신은 데이터 분석 전문 에이전트입니다.
    사용자가 데이터 분석을 요청하면, 적절한 Tool을 선택하여 실행하고
    결과를 한국어로 명확하게 해석해주십시오.
    항상 도구 실행 결과를 바탕으로 구체적인 수치와 함께 답변하십시오.
    """
)

# Tool 등록
@da_agent.tool
def tool_describe_data(ctx: RunContext[DataDeps]) -> dict:
    """데이터의 기본 통계량(형태, 수치형 요약, 범주형 요약)을 반환합니다."""
    return describe_data(ctx.deps.df)

@da_agent.tool
def tool_check_missing(ctx: RunContext[DataDeps]) -> dict:
    """각 컬럼의 결측치 개수와 비율을 분석합니다."""
    return check_missing(ctx.deps.df)

@da_agent.tool
def tool_detect_outliers(ctx: RunContext[DataDeps], column: str) -> dict:
    """지정한 수치형 컬럼에서 IQR 기준으로 이상치를 탐지합니다."""
    return detect_outliers(ctx.deps.df, column)

@da_agent.tool
def tool_plot_distribution(ctx: RunContext[DataDeps], column: str) -> dict:
    """지정한 컬럼의 분포를 시각화하고 요약 통계를 반환합니다."""
    return plot_distribution(ctx.deps.df, column)

@da_agent.tool
def tool_correlation_matrix(ctx: RunContext[DataDeps], method: str = 'pearson') -> dict:
    """수치형 컬럼 간의 상관관계 행렬을 계산하고 히트맵을 출력합니다."""
    return correlation_matrix(ctx.deps.df, method)

print("✅ DA 에이전트에 Tool 5종 등록 완료")
print("등록된 Tools:", [t.name for t in da_agent._function_tools.values()])

---
## 3. Tool 등록 에이전트 동작 확인

In [ ]:
# 에이전트가 Tool을 선택·실행하는지 확인
deps = DataDeps(df=df, csv_path="data/sample_data.csv")

async def test_tool_call():
    result = await da_agent.run(
        "이 데이터의 결측치 현황을 분석해주세요.",
        deps=deps
    )
    print("[에이전트 응답]")
    print(result.output)
    print()
    print("[Tool 호출 내역]")
    for msg in result.all_messages():
        print(f"  - {type(msg).__name__}")

await test_tool_call()

---
## 4. 정리

### 핵심 패턴
```python
@da_agent.tool
def my_tool(ctx: RunContext[DataDeps], param: str) -> dict:
    """Tool 설명 (에이전트가 이 docstring을 참고하여 Tool을 선택함)"""
    return 결과_딕셔너리
```

### 다음 노트북 (03)
CSV 파일 경로를 입력하면 에이전트가 **자율적으로** Tool을 선택·실행하여
전체 EDA 파이프라인을 수행하도록 연결한다.